# ⚡ Parallel Processing in Climatology Engine

This notebook introduces parallel processing methods to increase computational speed.

**What you will learn:**
- The concept of parallel processing and its benefits
- Different parallel processing methods in Python
- Using `multiprocessing` for parallel processing
- Using `joblib` for parallel processing
- Using `Dask` for parallel processing
- Comparing execution time across different methods
- Choosing the best method for large datasets

---

## 📐 Parallel Processing Theory

Parallel processing is a method to increase program execution speed by using multiple processor cores simultaneously.

### Benefits of Parallel Processing

| Benefit | Description |
|---------|-------------|
| **Higher Speed** | Reduced execution time by distributing work across processors |
| **Scalability** | Ability to process larger datasets with more cores |
| **Resource Optimization** | Maximum utilization of CPU power |
| **Better Responsiveness** | Concurrent execution of multiple tasks |

### Parallel Processing Methods in Python

| Method | Use Case | Advantages | Disadvantages |
|--------|----------|------------|---------------|
| `multiprocessing` | CPU-bound tasks | Full control over processes | Manual management |
| `joblib` | Large datasets | Simple and efficient | Library dependency |
| `Dask` | Very large datasets | High scalability | More complexity |
| `Ray` | Distributed systems | High flexibility | Separate installation |

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ Libraries loaded.')

In [ ]:
# Load distribution plugins
plugins = load_plugins()
print(f'✅ Number of loaded distributions: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# Load sample data
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# Select tmean data for one year
data_year = data[:365, 1]

print(f'📊 Number of samples: {len(data_year)}')
print(f'   Mean: {np.mean(data_year):.2f}°C')
print(f'   Standard deviation: {np.std(data_year):.2f}°C')

In [ ]:
# Define function for parallel processing
def process_station(station_data, dist_name='Normal'):
    """
    Process one station and fit distribution
    """
    dist = distributions[dist_name]
    try:
        res = dist.fit(station_data)
        return {
            'status': 'success',
            'aicc': res.get('aicc', np.nan),
            'bic': res.get('bic', np.nan),
            'loglik': res.get('loglik', np.nan)
        }
    except Exception as e:
        return {
            'status': 'error',
            'error': str(e)
        }

# Test the function on sample data
test_result = process_station(data_year, 'Normal')
print("✅ Processing function defined.")
print(f"   Test result: {test_result}")

In [ ]:
# ============================================================================
# 1. Serial Processing (No Parallelization)
# ============================================================================

def serial_processing(data_list, dist_name='Normal'):
    """Serial processing of data"""
    results = []
    for data in data_list:
        res = process_station(data, dist_name)
        results.append(res)
    return results

# Create test data (100 replicates of sample data with noise)
test_data = [data_year + np.random.normal(0, 0.1, len(data_year)) for _ in range(100)]

print(f"📊 Number of test datasets: {len(test_data)}")
print(f"   Size of each dataset: {len(test_data[0])}")

In [ ]:
# Run serial processing and measure time
print("\n🔄 Running serial processing...")
start_time = time.time()
serial_results = serial_processing(test_data, 'Normal')
serial_time = time.time() - start_time

success_count = sum(1 for r in serial_results if r.get('status') == 'success')
print(f"✅ Serial processing complete.")
print(f"   Execution time: {serial_time:.2f} seconds")
print(f"   Successful: {success_count} out of {len(test_data)}")

In [ ]:
# ============================================================================
# 2. Parallel Processing with multiprocessing
# ============================================================================

from multiprocessing import Pool, cpu_count

def parallel_processing_multiprocessing(data_list, dist_name='Normal', n_workers=None):
    """Parallel processing with multiprocessing"""
    if n_workers is None:
        n_workers = cpu_count()
    
    with Pool(processes=n_workers) as pool:
        results = pool.starmap(process_station, [(data, dist_name) for data in data_list])
    return results

print(f"✅ Available CPU cores: {cpu_count()}")

# Run parallel processing with multiprocessing
print("\n🔄 Running parallel processing with multiprocessing...")
start_time = time.time()
mp_results = parallel_processing_multiprocessing(test_data, 'Normal')
mp_time = time.time() - start_time

success_count = sum(1 for r in mp_results if r.get('status') == 'success')
print(f"✅ Parallel processing with multiprocessing complete.")
print(f"   Execution time: {mp_time:.2f} seconds")
print(f"   Successful: {success_count} out of {len(test_data)}")

In [ ]:
# ============================================================================
# 3. Parallel Processing with joblib
# ============================================================================

try:
    from joblib import Parallel, delayed
    
    def parallel_processing_joblib(data_list, dist_name='Normal', n_workers=-1):
        """Parallel processing with joblib"""
        if n_workers == -1:
            n_workers = cpu_count()
        
        results = Parallel(n_jobs=n_workers, verbose=0)(
            delayed(process_station)(data, dist_name) for data in data_list
        )
        return results

    print("\n🔄 Running parallel processing with joblib...")
    start_time = time.time()
    joblib_results = parallel_processing_joblib(test_data, 'Normal')
    joblib_time = time.time() - start_time

    success_count = sum(1 for r in joblib_results if r.get('status') == 'success')
    print(f"✅ Parallel processing with joblib complete.")
    print(f"   Execution time: {joblib_time:.2f} seconds")
    print(f"   Successful: {success_count} out of {len(test_data)}")
    
    joblib_available = True
except ImportError:
    print("⚠️ joblib library not installed. To install: pip install joblib")
    joblib_time = None
    joblib_available = False

In [ ]:
# ============================================================================
# 4. Parallel Processing with Dask
# ============================================================================

try:
    import dask
    from dask import delayed, compute
    import dask.multiprocessing
    
    def parallel_processing_dask(data_list, dist_name='Normal'):
        """Parallel processing with Dask"""
        lazy_results = [delayed(process_station)(data, dist_name) for data in data_list]
        results = compute(*lazy_results, scheduler='multiprocessing')
        return list(results)

    print("\n🔄 Running parallel processing with Dask...")
    start_time = time.time()
    dask_results = parallel_processing_dask(test_data, 'Normal')
    dask_time = time.time() - start_time

    success_count = sum(1 for r in dask_results if r.get('status') == 'success')
    print(f"✅ Parallel processing with Dask complete.")
    print(f"   Execution time: {dask_time:.2f} seconds")
    print(f"   Successful: {success_count} out of {len(test_data)}")
    
    dask_available = True
except ImportError:
    print("⚠️ Dask library not installed. To install: pip install dask")
    dask_time = None
    dask_available = False

In [ ]:
# ============================================================================
# 5. Execution Time Comparison
# ============================================================================

comparison_data = {
    'Method': ['Serial', 'multiprocessing', 'joblib', 'Dask'],
    'Time (s)': [serial_time, mp_time, joblib_time if joblib_available else np.nan, 
                 dask_time if dask_available else np.nan]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.dropna()

print("📊 Execution time comparison:")
print("=" * 60)
comparison_df

In [ ]:
# Plot execution time comparison
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']
bars = ax.bar(comparison_df['Method'], comparison_df['Time (s)'], 
              color=colors[:len(comparison_df)], alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('Processing Method', fontsize=12)
ax.set_ylabel('Execution Time (seconds)', fontsize=12)
ax.set_title('Execution Time Comparison Across Methods', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, comparison_df['Time (s)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{val:.2f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Calculate and display speedup
if len(comparison_df) > 1:
    serial_time = comparison_df.iloc[0]['Time (s)']
    for i in range(1, len(comparison_df)):
        speedup = serial_time / comparison_df.iloc[i]['Time (s)']
        ax.text(i, comparison_df.iloc[i]['Time (s)'] + 1, 
                f'Speedup: {speedup:.2f}x', ha='center', va='bottom', 
                fontsize=10, color='red')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 6. Effect of Number of Cores on Execution Time
# ============================================================================

def test_different_workers(data_list, dist_name='Normal'):
    """Test effect of number of workers on execution time"""
    n_workers_list = [1, 2, 4, 8]
    times = []
    
    for n in n_workers_list:
        try:
            if n > cpu_count():
                continue
            start_time = time.time()
            _ = parallel_processing_multiprocessing(data_list, dist_name, n_workers=n)
            elapsed = time.time() - start_time
            times.append((n, elapsed))
            print(f"   {n} cores: {elapsed:.2f} seconds")
        except Exception as e:
            print(f"   {n} cores: Error - {str(e)}")
    
    return times

print("\n📊 Effect of number of cores on execution time:")
print("=" * 50)
worker_times = test_different_workers(test_data[:50], 'Normal')

if worker_times:
    fig, ax = plt.subplots(figsize=(10, 6))
    workers = [w for w, _ in worker_times]
    times = [t for _, t in worker_times]
    
    ax.plot(workers, times, 'bo-', linewidth=2, markersize=10, 
            markeredgecolor='black', markeredgewidth=1)
    
    ax.set_xlabel('Number of Cores', fontsize=12)
    ax.set_ylabel('Execution Time (seconds)', fontsize=12)
    ax.set_title('Effect of Number of Cores on Execution Time', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    for x, y in zip(workers, times):
        ax.text(x, y + 0.1, f'{y:.2f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 📋 Summary

In this notebook you learned:

✅ The concept of parallel processing and its benefits
✅ Different parallel processing methods in Python
✅ Using `multiprocessing` for parallel processing
✅ Using `joblib` for parallel processing
✅ Using `Dask` for parallel processing
✅ Comparing execution time across different methods
✅ The effect of number of cores on execution time

---

**Key Takeaways:**

1. Parallel processing can reduce execution time by several times.
2. The choice of method depends on data type and computational complexity.
3. `multiprocessing` is suitable for fine-grained control over processes.
4. `joblib` is the simplest method to use.
5. `Dask` is suitable for very large datasets and distributed systems.
6. Increasing the number of cores doesn't always increase speed (communication overhead).

---

**Next Steps:**
- Notebook 08: Visualization
- Notebook 09: Custom Distribution
- Notebook 10: Advanced Usage